# Conversation state and scoring

Read `app/planner.py`, `app/interview.py`, and `app/analysis.py`. We provide extraction results explicitly, as a test would. There is no LLM or microphone in this notebook; it tests the controller, not recognition quality.

Run all cells in order. The deliberately small goal catalogue keeps the output understandable while exercising real code.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'app' / 'planner.py').exists()), None)
if root is None:
    raise RuntimeError('Open this notebook from the repository or docs/learning directory.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('Repository imports ready. No provider calls or application server will be started.')


Repository imports ready. No provider calls or application server will be started.


In [2]:
from app.models import AgentConfig, Extraction, Turn
from app.goals import GoalAnswer, MatchProfile, MatchRule
from app.interview import create_interview, apply_turn, stop_interview
from app.analysis import analyze
from app.errors import ServiceError
from uuid import uuid4

base = AgentConfig()
keys = {'startup_name','sector','stage','fundraising','funding_amount','next_step'}
cfg = AgentConfig(goals=[g for g in base.goals if g.key in keys], match_profile=MatchProfile(
    name='Fictional SaaS MVP profile', rules=[
        MatchRule(goal_key='sector',weight=3,operator='one_of',accepted=['SaaS']),
        MatchRule(goal_key='stage',weight=1,operator='one_of',accepted=['MVP']),
    ]))
item = create_interview('fictional-founder',cfg,'simulation')
def respond(key, text, status='answered', choice=None, number=None):
    turn = Turn(id=uuid4().hex,text=text,extraction=Extraction(answers=[GoalAnswer(
        goal_key=key,value=text,quote=text,status=status,choice_value=choice,number_value=number)]))
    apply_turn(item,turn)
    print('Next:', item.current_goal, '| Interview:', item.status)
    return turn
print(item.last_reply)
assert item.current_goal == 'startup_name'


Hi, ich bin Mira. Schön, dass du da bist. Wie heißt euer Startup?


## A legal suffix is not a complete name

The controller guards a known failure mode independently of model judgment. One bounded clarification is allowed. A later full name can replace the partial answer.


In [3]:
respond('startup_name','GmbH')
assert item.goals['startup_name'].status == 'partial'
assert item.goals['startup_name'].followups_used == 1
print(item.last_reply)
respond('startup_name','Nordlicht Energy Analytics GmbH')
assert item.current_goal == 'sector'


Next: startup_name | Interview: active
Ich habe nur die Rechtsform verstanden. Wie lautet der vollständige Firmenname?
Next: sector | Interview: active


## Match and coverage use different denominators

SaaS has weight 3. Stage has weight 1. When only SaaS is known and matches, all known evidence matches (100%), but only 75% of the configured weight is known.


In [4]:
last_turn = respond('sector','Wir entwickeln SaaS.',choice='SaaS')
report = analyze(item)
print({k:report[k] for k in ['score','coverage','completeness']})
assert report['score'] == 100 and report['coverage'] == 75
respond('stage','Wir haben einen Prototyp.',choice='Prototyp')
report = analyze(item)
print({k:report[k] for k in ['score','coverage']})
assert report['score'] == 75 and report['coverage'] == 100


Next: stage | Interview: active
{'score': 100, 'coverage': 75, 'completeness': 33}
Next: fundraising | Interview: active
{'score': 75, 'coverage': 100}


## Conditional questions and summary confirmation

Declining fundraising excludes the funding amount. Completing all applicable goals enters review; it does not automatically mark the interview completed.


In [5]:
respond('fundraising','Wir planen keine Runde.',choice='Nein')
assert item.current_goal == 'next_step'
report = analyze(item)
assert next(g for g in report['goals'] if g['key']=='funding_amount')['status']=='not_applicable'
respond('next_step','Wir wünschen uns ein Folgegespräch.')
assert item.status == 'review'
print(item.last_reply)


Next: next_step | Interview: active
Next: None | Interview: review
Startup-Name: Nordlicht Energy Analytics GmbH. Sektor: Wir entwickeln SaaS.. Entwicklungsphase: Wir haben einen Prototyp.. Finanzierung geplant: Wir planen keine Runde.. Nächster Schritt: Wir wünschen uns ein Folgegespräch.. Stimmt das, oder möchtest du etwas korrigieren?


## Corrections take precedence over confirmation

A correction and “yes” in the same turn require a new review. The planner never treats approval as permission to ignore newly changed information.


In [6]:
correction = Turn(id=uuid4().hex,text='Wir haben inzwischen ein MVP.',extraction=Extraction(
    answers=[GoalAnswer(goal_key='stage',status='answered',value='MVP',quote='ein MVP',choice_value='MVP')],
    confirmed_summary=True))
apply_turn(item,correction)
assert item.status == 'review'
assert analyze(item)['score'] == 100
confirmation = Turn(id=uuid4().hex,text='Ja, das stimmt.',extraction=Extraction(confirmed_summary=True))
apply_turn(item,confirmation)
assert item.status == 'completed'
print(item.status, '|', item.last_reply)


completed | Danke! Dein Gesprächsergebnis ist gespeichert.


## Repeated requests and invalid evidence

The same ID and payload is a retry, not a new answer. A quote that never occurred in the turn is rejected before changing goal state. These checks do not establish that an existing quote semantically supports the claimed value.


In [7]:
revision = item.revision
apply_turn(item,confirmation)
assert item.revision == revision
print('Duplicate confirmation left revision unchanged:', revision)
fresh = create_interview('another-founder',cfg,'simulation')
bad = Turn(id='bad-evidence',text='Hallo',extraction=Extraction(answers=[GoalAnswer(
    goal_key='startup_name',status='answered',value='Invented',quote='never said')]))
try:
    apply_turn(fresh,bad)
except ServiceError as exc:
    assert exc.code == 'INVALID_EVIDENCE'
    print('Rejected:', exc.code)
assert fresh.goals['startup_name'].value is None
stop_interview(fresh)
assert fresh.status == 'incomplete'
print('Stopping without confirmation:', fresh.status)


Duplicate confirmation left revision unchanged: 8
Rejected: INVALID_EVIDENCE
Stopping without confirmation: incomplete


## Exercises

- Change the stage rule's weight to 3 and recompute the partial-evidence example by hand.
- Repeat a vague answer twice and inspect `exhausted` and `followups_used`.
- Answer a future goal before the current question. The controller should retain it and avoid asking it again.
- Explain why a completed interview can still contain unknown answers.
- The transcript here has user contributions only: the real worker separately persists spoken assistant contributions. Simulation HTTP routes also append assistant replies; calling the domain function directly does not do that.
